# 02 - Provider 对比测试

**目的**: 直接对比各 LLM Provider 的延迟、Token 消耗和成本  
**测试对象**: APIPro (Claude / Qwen / DeepSeek) + Ollama (本地模型)  
**调用方式**: 直接 HTTP 调用，绕过 AI Engine，测量纯 Provider 性能  
**输出**: Provider 对比表格 + 成本估算 + 选型建议

**注意**: 需要在 `.env` 中配置 `APIPRO_API_KEY`

## 0. 配置参数（修改此处进行实验）

In [ ]:
import os
from datetime import datetime

# =============================================
# Provider 配置
# =============================================

APIPRO_BASE_URL = os.getenv("APIPRO_BASE_URL", "https://api.apipro.ai/v1")
APIPRO_API_KEY = os.getenv("APIPRO_API_KEY", "")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# 要测试的 Provider 列表（可增删）
PROVIDERS = [
    {
        "id": "apipro_claude",
        "label": "APIPro Claude Sonnet",
        "type": "apipro",
        "model": "claude-sonnet-4-5",
        "cost_per_1k_input": 0.003,
        "cost_per_1k_output": 0.015,
        "enabled": bool(APIPRO_API_KEY)
    },
    {
        "id": "apipro_qwen",
        "label": "APIPro Qwen Plus",
        "type": "apipro",
        "model": "qwen-plus",
        "cost_per_1k_input": 0.0008,
        "cost_per_1k_output": 0.002,
        "enabled": bool(APIPRO_API_KEY)
    },
    {
        "id": "apipro_deepseek",
        "label": "APIPro DeepSeek Chat",
        "type": "apipro",
        "model": "deepseek-chat",
        "cost_per_1k_input": 0.00014,
        "cost_per_1k_output": 0.00028,
        "enabled": bool(APIPRO_API_KEY)
    },
    {
        "id": "ollama_qwen14b",
        "label": "Ollama Qwen2.5:14b",
        "type": "ollama",
        "model": "qwen2.5:14b",
        "cost_per_1k_input": 0.0,
        "cost_per_1k_output": 0.0,
        "enabled": True
    },
    {
        "id": "ollama_qwen7b",
        "label": "Ollama Qwen2.5:7b",
        "type": "ollama",
        "model": "qwen2.5:7b",
        "cost_per_1k_input": 0.0,
        "cost_per_1k_output": 0.0,
        "enabled": True
    }
]

# 每个 Provider 测试轮数
N_ITERATIONS = 5

# 超时设置（秒）
TIMEOUT_APIPRO = 30
TIMEOUT_OLLAMA = 120

# 结果目录
RESULTS_DIR = "../data"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = f"{RESULTS_DIR}/provider_comparison_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

enabled_providers = [p for p in PROVIDERS if p["enabled"]]
print(f"配置完成")
print(f"APIPro Key: {'已配置' if APIPRO_API_KEY else '未配置 - APIPro Provider 将跳过'}")
print(f"启用的 Provider: {[p['label'] for p in enabled_providers]}")
print(f"每个 Provider 测试轮数: {N_ITERATIONS}")

## 1. 依赖导入

In [ ]:
import httpx
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. 测试 Prompt（可修改）

In [ ]:
# 测试场景 Prompt（贴近真实业务）
TEST_SCENARIOS = [
    {
        "id": "requirement_extraction",
        "label": "需求提取",
        "system": "你是一个采购助手，帮助用户整理采购需求。请从用户输入中提取：项目名称、品类、数量、预算、交货日期。以JSON格式返回。",
        "user": "我要采购100台服务器，Dell品牌，用于数据中心，预算500万，3月底前到货"
    },
    {
        "id": "simple_qa",
        "label": "简单问答",
        "system": "你是一个采购助手。",
        "user": "请问采购流程一般包括哪些步骤？"
    }
]

# 本次测试使用的场景（可改为 TEST_SCENARIOS[1] 等）
ACTIVE_SCENARIO = TEST_SCENARIOS[0]

print(f"当前测试场景: {ACTIVE_SCENARIO['label']}")
print(f"System: {ACTIVE_SCENARIO['system'][:80]}...")
print(f"User: {ACTIVE_SCENARIO['user']}")

## 3. Provider 调用函数

In [ ]:
def call_apipro(model: str, system: str, user: str, timeout: int = 30) -> dict:
    """调用 APIPro（OpenAI 兼容格式）"""
    start = time.perf_counter()
    try:
        resp = httpx.post(
            f"{APIPRO_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {APIPRO_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ],
                "max_tokens": 512,
                "temperature": 0.7
            },
            timeout=timeout
        )
        elapsed = (time.perf_counter() - start) * 1000

        if resp.status_code == 200:
            data = resp.json()
            usage = data.get("usage", {})
            return {
                "success": True,
                "latency_ms": elapsed,
                "input_tokens": usage.get("prompt_tokens", 0),
                "output_tokens": usage.get("completion_tokens", 0),
                "content": data["choices"][0]["message"]["content"],
                "error": None
            }
        else:
            return {"success": False, "latency_ms": elapsed, "input_tokens": 0,
                    "output_tokens": 0, "content": None, "error": f"HTTP {resp.status_code}: {resp.text[:200]}"}
    except Exception as e:
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": False, "latency_ms": elapsed, "input_tokens": 0,
                "output_tokens": 0, "content": None, "error": str(e)}


def call_ollama(model: str, system: str, user: str, timeout: int = 120) -> dict:
    """调用本地 Ollama"""
    start = time.perf_counter()
    try:
        # 先检查模型是否存在
        tags_resp = httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        if tags_resp.status_code != 200:
            return {"success": False, "latency_ms": 0, "input_tokens": 0,
                    "output_tokens": 0, "content": None, "error": "Ollama 服务不可用"}

        available_models = [m["name"] for m in tags_resp.json().get("models", [])]
        model_available = any(model in m for m in available_models)
        if not model_available:
            return {"success": False, "latency_ms": 0, "input_tokens": 0,
                    "output_tokens": 0, "content": None,
                    "error": f"模型 {model} 未安装。可用: {available_models}"}

        # 使用 chat 格式
        resp = httpx.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": model,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ],
                "stream": False
            },
            timeout=timeout
        )
        elapsed = (time.perf_counter() - start) * 1000

        if resp.status_code == 200:
            data = resp.json()
            prompt_eval = data.get("prompt_eval_count", 0)
            eval_count = data.get("eval_count", 0)
            return {
                "success": True,
                "latency_ms": elapsed,
                "input_tokens": prompt_eval,
                "output_tokens": eval_count,
                "content": data["message"]["content"],
                "error": None
            }
        else:
            return {"success": False, "latency_ms": elapsed, "input_tokens": 0,
                    "output_tokens": 0, "content": None, "error": resp.text[:200]}
    except Exception as e:
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": False, "latency_ms": elapsed, "input_tokens": 0,
                "output_tokens": 0, "content": None, "error": str(e)}


print("调用函数定义完成")

## 4. 快速单次测试（验证连通性）

In [ ]:
print("快速连通性验证（每个 Provider 各发一条请求）")
print("=" * 60)

for provider in enabled_providers:
    if provider["type"] == "apipro":
        result = call_apipro(
            provider["model"],
            ACTIVE_SCENARIO["system"],
            ACTIVE_SCENARIO["user"],
            timeout=TIMEOUT_APIPRO
        )
    else:
        result = call_ollama(
            provider["model"],
            ACTIVE_SCENARIO["system"],
            ACTIVE_SCENARIO["user"],
            timeout=TIMEOUT_OLLAMA
        )

    status = "OK" if result["success"] else "FAIL"
    latency = f"{result['latency_ms']:.0f}ms" if result["success"] else "N/A"
    tokens = f"in={result['input_tokens']} out={result['output_tokens']}" if result["success"] else ""
    error = f"| {result['error'][:60]}" if result["error"] else ""
    print(f"[{status}] {provider['label']:30s} {latency:8s} {tokens} {error}")

## 5. 批量对比测试

In [ ]:
import time as time_module

all_results = []

print(f"开始批量测试: {len(enabled_providers)} 个 Provider × {N_ITERATIONS} 轮")
print("=" * 60)

for provider in enabled_providers:
    print(f"\n测试: {provider['label']}")
    for i in range(N_ITERATIONS):
        if provider["type"] == "apipro":
            result = call_apipro(
                provider["model"],
                ACTIVE_SCENARIO["system"],
                ACTIVE_SCENARIO["user"],
                timeout=TIMEOUT_APIPRO
            )
        else:
            result = call_ollama(
                provider["model"],
                ACTIVE_SCENARIO["system"],
                ACTIVE_SCENARIO["user"],
                timeout=TIMEOUT_OLLAMA
            )

        # 计算成本
        cost_usd = (
            result["input_tokens"] / 1000 * provider["cost_per_1k_input"] +
            result["output_tokens"] / 1000 * provider["cost_per_1k_output"]
        ) if result["success"] else 0

        record = {
            "provider_id": provider["id"],
            "provider_label": provider["label"],
            "model": provider["model"],
            "provider_type": provider["type"],
            "iteration": i + 1,
            "success": result["success"],
            "latency_ms": result["latency_ms"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "total_tokens": result["input_tokens"] + result["output_tokens"],
            "cost_usd": cost_usd,
            "error": result["error"],
            "timestamp": datetime.now().isoformat()
        }
        all_results.append(record)

        status = "OK" if result["success"] else "FAIL"
        print(f"  [{status}] 轮{i+1} | {result['latency_ms']:6.0f}ms | tokens={result['input_tokens']}+{result['output_tokens']} | ${cost_usd:.5f}")

        if i < N_ITERATIONS - 1:
            time_module.sleep(0.5)

df = pd.DataFrame(all_results)
print(f"\n测试完成，共 {len(df)} 条记录")

## 6. 统计对比

In [ ]:
df_ok = df[df["success"] == True].copy()

print("=" * 70)
print("Provider 对比统计")
print("=" * 70)

summary = df_ok.groupby("provider_label").agg(
    成功次数=("success", "count"),
    P50延迟=("latency_ms", lambda x: np.percentile(x, 50)),
    P95延迟=("latency_ms", lambda x: np.percentile(x, 95)),
    平均延迟=("latency_ms", "mean"),
    平均输入Token=("input_tokens", "mean"),
    平均输出Token=("output_tokens", "mean"),
    平均成本USD=("cost_usd", "mean")
).round(2)

print(summary.to_string())

print("\n成功率:")
for p in enabled_providers:
    total = len(df[df["provider_id"] == p["id"]])
    success = len(df[(df["provider_id"] == p["id"]) & (df["success"] == True)])
    print(f"  {p['label']:30s} {success}/{total} ({100*success/total:.0f}%)")

## 7. 可视化对比

In [ ]:
if len(df_ok) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Provider 对比 - {ACTIVE_SCENARIO['label']}", fontsize=13)

    labels = df_ok["provider_label"].unique().tolist()

    # 图1: 延迟对比箱线图
    ax1 = axes[0]
    data_by_provider = [df_ok[df_ok["provider_label"] == l]["latency_ms"].values for l in labels]
    ax1.boxplot(data_by_provider, labels=[l.split()[-1] for l in labels])
    ax1.set_title("延迟对比 (ms)")
    ax1.set_ylabel("延迟 (ms)")
    ax1.tick_params(axis='x', rotation=20)

    # 图2: Token 消耗对比
    ax2 = axes[1]
    avg_input = [df_ok[df_ok["provider_label"] == l]["input_tokens"].mean() for l in labels]
    avg_output = [df_ok[df_ok["provider_label"] == l]["output_tokens"].mean() for l in labels]
    x = range(len(labels))
    width = 0.35
    ax2.bar([i - width/2 for i in x], avg_input, width, label="输入 Token", color="steelblue")
    ax2.bar([i + width/2 for i in x], avg_output, width, label="输出 Token", color="coral")
    ax2.set_title("平均 Token 消耗")
    ax2.set_ylabel("Token 数量")
    ax2.set_xticks(list(x))
    ax2.set_xticklabels([l.split()[-1] for l in labels], rotation=20)
    ax2.legend()

    # 图3: 成本对比（仅 APIPro）
    ax3 = axes[2]
    avg_cost = [df_ok[df_ok["provider_label"] == l]["cost_usd"].mean() * 1000 for l in labels]  # 转换为 per 1000 requests
    colors = ["steelblue" if c > 0 else "lightgreen" for c in avg_cost]
    ax3.bar(range(len(labels)), avg_cost, color=colors)
    ax3.set_title("单次请求成本 (USD × 1000)")
    ax3.set_ylabel("成本 (mUSD)")
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels([l.split()[-1] for l in labels], rotation=20)

    plt.tight_layout()
    chart_file = f"{RESULTS_DIR}/provider_comparison_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
    plt.savefig(chart_file, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图表已保存: {chart_file}")
else:
    print("无成功数据，跳过可视化")

## 8. 选型建议

In [ ]:
if len(df_ok) > 0:
    print("=" * 60)
    print("选型建议（基于测试数据）")
    print("=" * 60)

    # 最快
    fastest = df_ok.groupby("provider_label")["latency_ms"].mean().idxmin()
    fastest_ms = df_ok.groupby("provider_label")["latency_ms"].mean().min()

    # 最便宜（非免费）
    paid = df_ok[df_ok["cost_usd"] > 0]
    if len(paid) > 0:
        cheapest = paid.groupby("provider_label")["cost_usd"].mean().idxmin()
        cheapest_cost = paid.groupby("provider_label")["cost_usd"].mean().min()
    else:
        cheapest = "Ollama (免费)"
        cheapest_cost = 0

    print(f"\n速度最快: {fastest} ({fastest_ms:.0f}ms P50)")
    print(f"成本最低: {cheapest} (${cheapest_cost:.5f}/请求)")

    print("\n场景推荐:")
    print("  开发/测试: DeepSeek Chat 或 Ollama (低成本，够用)")
    print("  生产-简单: DeepSeek Chat (低成本，快速)")
    print("  生产-复杂: Claude Sonnet (质量优先)")
    print("  离线/备用: Ollama Qwen2.5:14b")

## 9. 保存结果

In [ ]:
df.to_csv(RESULTS_FILE, index=False, encoding="utf-8")
print(f"结果已保存: {RESULTS_FILE}")
df.head(10)